# AI Use Case 2 — Pydantic: ferramentas (tool calling) para agentes

**Problema:** num agente, o LLM decide chamar funções do seu sistema (`buscar_pedido`, `emitir_reembolso`...).
Os argumentos vêm **do modelo** — ou seja, de uma fonte não confiável. Você precisa: descrever cada ferramenta com um
schema, validar os argumentos antes de executar, devolver erros de forma que o modelo consiga se corrigir e impor limites
de segurança (*guardrails*) que nenhum prompt garante.

**Ideia central:** um modelo Pydantic por ferramenta é, ao mesmo tempo, **a documentação para o LLM** (JSON Schema),
**o validador dos argumentos** e **o guardrail**.

```mermaid
sequenceDiagram
    participant U as Usuário
    participant A as Loop do agente
    participant L as LLM
    participant R as ToolRegistry (Pydantic)
    participant S as Sistema
    U->>A: "quero reembolso do pedido 123"
    A->>L: mensagens + specs das ferramentas (JSON Schema)
    L-->>A: tool_use(get_order, {order_id: "123"})
    A->>R: validar args com GetOrderArgs
    R->>S: get_order(GetOrderArgs)
    S-->>A: OrderInfo (Pydantic) → JSON
    A->>L: tool_result
    L-->>A: tool_use(issue_refund, {amount: 5000})
    A->>R: validar → guardrail: amount > total!
    R-->>A: is_error + mensagem de validação
    A->>L: tool_result (erro)
    L-->>A: tool_use(issue_refund, {amount: 189.9}) ✓
```

Roteiro:
1. Um registro de ferramentas baseado em Pydantic
2. Specs para os provedores (formato Anthropic e OpenAI)
3. O loop do agente com validação, erros devolvidos ao modelo e guardrails
4. Ferramentas a partir da assinatura da função (`create_model`)
5. Resultados de ferramentas também com contrato
6. Checklist

In [1]:
import inspect, json
from dataclasses import dataclass
from decimal import Decimal
from typing import Annotated, Any, Callable, Literal
from pydantic import BaseModel, ConfigDict, Field, ValidationError, ValidationInfo, create_model, field_validator
from fake_llm import ScriptedLLM, LLMResponse, ToolCall

## 1. Registro de ferramentas

O decorador `@registry.tool(ArgsModel)` associa a função ao seu modelo de argumentos. O **docstring** vira a descrição da ferramenta;
as `description` dos campos viram a documentação de cada argumento.

In [2]:
class ToolArgs(BaseModel):
    """Base para argumentos de ferramentas: nada de campos extras inventados pelo modelo."""
    model_config = ConfigDict(extra="forbid")

@dataclass(frozen=True)
class Tool:
    name: str
    description: str
    args_model: type[ToolArgs]
    fn: Callable[..., Any]

class ToolRegistry:
    def __init__(self):
        self._tools: dict[str, Tool] = {}

    def tool(self, args_model: type[ToolArgs], name: str | None = None):
        def register(fn):
            t = Tool(name or fn.__name__, inspect.getdoc(fn) or "", args_model, fn)
            self._tools[t.name] = t
            return fn
        return register

    def add(self, name: str, description: str, args_model: type[ToolArgs], fn: Callable[..., Any]) -> None:
        self._tools[name] = Tool(name, description, args_model, fn)

    # ---- specs para os provedores ----
    def _schema(self, t: Tool) -> dict:
        s = t.args_model.model_json_schema()
        s.pop("title", None); s.pop("description", None)
        return s

    def anthropic_specs(self) -> list[dict]:
        return [{"name": t.name, "description": t.description, "input_schema": self._schema(t), "strict": True}
                for t in self._tools.values()]

    def openai_specs(self) -> list[dict]:
        return [{"type": "function", "name": t.name, "description": t.description,
                 "parameters": self._schema(t), "strict": True} for t in self._tools.values()]

    # ---- execução ----
    def execute(self, call: ToolCall, context: dict | None = None) -> tuple[str, bool]:
        """Devolve (conteúdo, is_error). Nunca levanta: erros viram mensagem para o modelo."""
        tool = self._tools.get(call.name)
        if tool is None:
            return f"Ferramenta desconhecida: {call.name}. Disponíveis: {sorted(self._tools)}", True
        try:
            args = tool.args_model.model_validate(call.input, context=context)
        except ValidationError as e:
            msgs = [f"{'.'.join(map(str, err['loc'])) or '(args)'}: {err['msg']}" for err in e.errors(include_url=False)]
            return "Argumentos inválidos:\n" + "\n".join(msgs), True
        try:
            result = tool.fn(args)
        except LookupError as e:                       # erros de negócio esperados
            return str(e), True
        return result.model_dump_json() if isinstance(result, BaseModel) else json.dumps(result, default=str, ensure_ascii=False), False

registry = ToolRegistry()

### Ferramentas de um agente de suporte de e-commerce
Repare no guardrail de `issue_refund`: o limite vem do **contexto** de execução (o pedido real), não do prompt.

In [3]:
ORDERS = {
    "123": {"order_id": "123", "customer": "ana@x.com", "total": Decimal("189.90"), "status": "delivered", "refunded": Decimal("0")},
    "456": {"order_id": "456", "customer": "bia@x.com", "total": Decimal("59.00"), "status": "shipped", "refunded": Decimal("0")},
}

class OrderInfo(BaseModel):
    order_id: str
    customer: str
    total: Decimal
    status: Literal["paid", "shipped", "delivered", "cancelled"]
    refunded: Decimal

class GetOrderArgs(ToolArgs):
    order_id: str = Field(pattern=r"^\d{3,10}$", description="Número do pedido, só dígitos")

@registry.tool(GetOrderArgs)
def get_order(args: GetOrderArgs) -> OrderInfo:
    """Busca um pedido pelo número. Use antes de qualquer ação sobre o pedido."""
    if args.order_id not in ORDERS:
        raise LookupError(f"pedido {args.order_id} não encontrado")
    return OrderInfo(**ORDERS[args.order_id])

from pydantic import WithJsonSchema

# Decimal gera um schema "anyOf number|string com regex" — ruim para LLMs em modo estrito.
# WithJsonSchema publica um schema simples; a validação local continua sendo Decimal com 2 casas.
MoneyArg = Annotated[Decimal, Field(gt=0, decimal_places=2), WithJsonSchema({"type": "number", "exclusiveMinimum": 0})]

class RefundArgs(ToolArgs):
    order_id: str = Field(pattern=r"^\d{3,10}$")
    amount: MoneyArg = Field(description="Valor do reembolso em reais, com 2 casas")
    reason: Literal["damaged", "not_delivered", "wrong_item", "customer_regret"] = Field(
        description="Motivo do reembolso")

    @field_validator("amount")
    @classmethod
    def within_order_total(cls, v: Decimal, info: ValidationInfo) -> Decimal:
        order = ORDERS.get(info.data.get("order_id", ""))
        if order:
            available = order["total"] - order["refunded"]
            if v > available:
                raise ValueError(f"valor {v} excede o reembolsável do pedido ({available})")
        limit = (info.context or {}).get("max_refund_without_human")
        if limit is not None and v > limit:
            raise ValueError(f"valor acima de {limit} exige aprovação humana: use escalate_to_human")
        return v

@registry.tool(RefundArgs)
def issue_refund(args: RefundArgs) -> dict:
    """Emite reembolso (total ou parcial) de um pedido entregue ou enviado."""
    ORDERS[args.order_id]["refunded"] += args.amount
    return {"ok": True, "order_id": args.order_id, "refunded": args.amount, "reason": args.reason}

class EscalateArgs(ToolArgs):
    summary: str = Field(min_length=20, description="Resumo do caso para o atendente humano")
    priority: Literal["low", "normal", "high"] = "normal"

@registry.tool(EscalateArgs)
def escalate_to_human(args: EscalateArgs) -> dict:
    """Encaminha o caso para um atendente humano quando a política não permite resolver automaticamente."""
    return {"ticket": "HUM-0001", "priority": args.priority}

## 2. Specs para os provedores

In [4]:
print(f"json.dumps(registry.anthropic_specs()[1], indent=2, ensure_ascii=False):\n{json.dumps(registry.anthropic_specs()[1], indent=2, ensure_ascii=False)}")

json.dumps(registry.anthropic_specs()[1], indent=2, ensure_ascii=False):
{
  "name": "issue_refund",
  "description": "Emite reembolso (total ou parcial) de um pedido entregue ou enviado.",
  "input_schema": {
    "additionalProperties": false,
    "properties": {
      "order_id": {
        "pattern": "^\\d{3,10}$",
        "title": "Order Id",
        "type": "string"
      },
      "amount": {
        "description": "Valor do reembolso em reais, com 2 casas",
        "exclusiveMinimum": 0,
        "title": "Amount",
        "type": "number"
      },
      "reason": {
        "description": "Motivo do reembolso",
        "enum": [
          "damaged",
          "not_delivered",
          "wrong_item",
          "customer_regret"
        ],
        "title": "Reason",
        "type": "string"
      }
    },
    "required": [
      "order_id",
      "amount",
      "reason"
    ],
    "type": "object"
  },
  "strict": true
}


> **Por que o `MoneyArg`?** Sem ele, `Decimal` vira `{"anyOf": [{"type": "number"}, {"type": "string", "pattern": "<regex longa>"}]}`.
> Schemas para LLM devem ser **simples**: uniões e regex complexas confundem o modelo e alguns provedores, em modo estrito,
> rejeitam ou ignoram construções avançadas. Separe o **schema publicado** (simples) da **validação local** (rigorosa).

In [5]:
print(f"json.dumps(registry.openai_specs()[0], indent=2, ensure_ascii=False):\n{json.dumps(registry.openai_specs()[0], indent=2, ensure_ascii=False)}")

json.dumps(registry.openai_specs()[0], indent=2, ensure_ascii=False):
{
  "type": "function",
  "name": "get_order",
  "description": "Busca um pedido pelo número. Use antes de qualquer ação sobre o pedido.",
  "parameters": {
    "additionalProperties": false,
    "properties": {
      "order_id": {
        "description": "Número do pedido, só dígitos",
        "pattern": "^\\d{3,10}$",
        "title": "Order Id",
        "type": "string"
      }
    },
    "required": [
      "order_id"
    ],
    "type": "object"
  },
  "strict": true
}


Uso real (referência, não executado):

```python
# Anthropic
response = client.messages.create(model="claude-opus-5-5", max_tokens=1024,
                                  tools=registry.anthropic_specs(), messages=messages)
for block in response.content:
    if block.type == "tool_use":
        content, is_error = registry.execute(ToolCall(block.id, block.name, block.input))
        # devolver {"type": "tool_result", "tool_use_id": block.id, "content": content, "is_error": is_error}
```

Com `"strict": true` o provedor garante que `input` segue o schema (tipos, enums, campos obrigatórios). A validação local
continua necessária para **regras de negócio** (`within_order_total`) e restrições que o modo estrito não impõe.

## 3. O loop do agente

O loop: chama o LLM → executa as ferramentas pedidas → devolve os resultados → repete até o modelo responder sem pedir ferramenta.
O roteiro abaixo simula um modelo que **erra o valor** do reembolso e se corrige ao ler o erro de validação.

In [6]:
def run_agent(llm, user_msg: str, context: dict, max_steps: int = 6) -> str:
    messages = [{"role": "user", "content": user_msg}]
    for step in range(max_steps):
        resp = llm(messages, tools=registry.anthropic_specs())
        if not resp.tool_calls:
            return resp.text
        messages.append({"role": "assistant", "content": [
            {"type": "tool_use", "id": c.id, "name": c.name, "input": c.input} for c in resp.tool_calls]})
        results = []
        for call in resp.tool_calls:
            content, is_error = registry.execute(call, context=context)
            flag = "ERRO" if is_error else "ok"
            print(f"[passo {step}] {call.name}({call.input}) -> {flag}: {content}")
            results.append({"type": "tool_result", "tool_use_id": call.id, "content": content, "is_error": is_error})
        messages.append({"role": "user", "content": results})
    raise RuntimeError("agente excedeu o número máximo de passos")

script = [
    LLMResponse(tool_calls=(ToolCall("t1", "get_order", {"order_id": "123"}),)),
    LLMResponse(tool_calls=(ToolCall("t2", "issue_refund", {"order_id": "123", "amount": "1899.00", "reason": "damaged"}),)),
    LLMResponse(tool_calls=(ToolCall("t3", "issue_refund", {"order_id": "123", "amount": "189.90", "reason": "damaged"}),)),
    LLMResponse(text="Pronto! Reembolsei R$ 189,90 do pedido 123. O valor aparece em até 5 dias úteis."),
]
answer = run_agent(ScriptedLLM(script), "O pedido 123 chegou quebrado, quero meu dinheiro de volta.",
                   context={"max_refund_without_human": Decimal("500")})
print(f"\nresposta final: {answer}")
print(f'estado do pedido: {ORDERS["123"]["refunded"]}')

[passo 0] get_order({'order_id': '123'}) -> ok: {"order_id":"123","customer":"ana@x.com","total":"189.90","status":"delivered","refunded":"0"}
[passo 1] issue_refund({'order_id': '123', 'amount': '1899.00', 'reason': 'damaged'}) -> ERRO: Argumentos inválidos:
amount: Value error, valor 1899.00 excede o reembolsável do pedido (189.90)
[passo 2] issue_refund({'order_id': '123', 'amount': '189.90', 'reason': 'damaged'}) -> ok: {"ok": true, "order_id": "123", "refunded": "189.90", "reason": "damaged"}

resposta final: Pronto! Reembolsei R$ 189,90 do pedido 123. O valor aparece em até 5 dias úteis.
estado do pedido: 189.90


Agora com um **limite de política** menor no contexto: o modelo tenta reembolsar, é barrado e escala para humano.
Note que o modelo também tenta inventar um argumento (`notify`) — `extra="forbid"` barra.

In [7]:
script = [
    LLMResponse(tool_calls=(ToolCall("t1", "issue_refund", {"order_id": "456", "amount": "59.00", "reason": "not_delivered", "notify": True}),)),
    LLMResponse(tool_calls=(ToolCall("t2", "issue_refund", {"order_id": "456", "amount": "59.00", "reason": "not_delivered"}),)),
    LLMResponse(tool_calls=(ToolCall("t3", "escalate_to_human", {"summary": "Cliente bia@x.com: pedido 456 não entregue, pede reembolso de R$ 59,00", "priority": "high"}),)),
    LLMResponse(text="Encaminhei seu caso (HUM-0001) para um atendente, que vai finalizar o reembolso."),
]
answer = run_agent(ScriptedLLM(script), "Meu pedido 456 nunca chegou", context={"max_refund_without_human": Decimal("50")})
print(f"\nresposta final: {answer}")

[passo 0] issue_refund({'order_id': '456', 'amount': '59.00', 'reason': 'not_delivered', 'notify': True}) -> ERRO: Argumentos inválidos:
amount: Value error, valor acima de 50 exige aprovação humana: use escalate_to_human
notify: Extra inputs are not permitted
[passo 1] issue_refund({'order_id': '456', 'amount': '59.00', 'reason': 'not_delivered'}) -> ERRO: Argumentos inválidos:
amount: Value error, valor acima de 50 exige aprovação humana: use escalate_to_human
[passo 2] escalate_to_human({'summary': 'Cliente bia@x.com: pedido 456 não entregue, pede reembolso de R$ 59,00', 'priority': 'high'}) -> ok: {"ticket": "HUM-0001", "priority": "high"}

resposta final: Encaminhei seu caso (HUM-0001) para um atendente, que vai finalizar o reembolso.


## 4. Ferramentas a partir da assinatura da função

Para muitas ferramentas pequenas, escrever um modelo por ferramenta é verboso. `create_model` monta o modelo a partir da
assinatura (type hints + defaults + `Annotated[..., Field(...)]`). É, essencialmente, o que frameworks de agentes fazem por baixo.

In [8]:
from typing import Annotated, get_type_hints

def args_model_from_signature(fn: Callable) -> type[ToolArgs]:
    hints = get_type_hints(fn, include_extras=True)
    fields = {}
    for name, param in inspect.signature(fn).parameters.items():
        default = ... if param.default is inspect.Parameter.empty else param.default
        fields[name] = (hints[name], default)
    return create_model(f"{fn.__name__}_args", __base__=ToolArgs, **fields)

def track_shipment(tracking_code: Annotated[str, Field(pattern=r"^[A-Z]{2}\d{9}BR$", description="Código dos Correios")],
                   include_history: bool = False) -> dict:
    """Consulta o rastreamento de um envio."""
    return {"code": tracking_code, "status": "em trânsito", "history": ["postado"] if include_history else []}

Model = args_model_from_signature(track_shipment)
registry.add("track_shipment", inspect.getdoc(track_shipment), Model,
             lambda args: track_shipment(**args.model_dump()))

print(f"json.dumps(registry.anthropic_specs()[-1], indent=2, ensure_ascii=False):\n{json.dumps(registry.anthropic_specs()[-1], indent=2, ensure_ascii=False)}")
for inp in ({"tracking_code": "AB123456789BR", "include_history": True}, {"tracking_code": "123"}):
    print(f'registry.execute(ToolCall("x", "track_shipment", inp)): {registry.execute(ToolCall("x", "track_shipment", inp))}')

json.dumps(registry.anthropic_specs()[-1], indent=2, ensure_ascii=False):
{
  "name": "track_shipment",
  "description": "Consulta o rastreamento de um envio.",
  "input_schema": {
    "additionalProperties": false,
    "properties": {
      "tracking_code": {
        "description": "Código dos Correios",
        "pattern": "^[A-Z]{2}\\d{9}BR$",
        "title": "Tracking Code",
        "type": "string"
      },
      "include_history": {
        "default": false,
        "title": "Include History",
        "type": "boolean"
      }
    },
    "required": [
      "tracking_code"
    ],
    "type": "object"
  },
  "strict": true
}
registry.execute(ToolCall("x", "track_shipment", inp)): ('{"code": "AB123456789BR", "status": "em trânsito", "history": ["postado"]}', False)
registry.execute(ToolCall("x", "track_shipment", inp)): ("Argumentos inválidos:\ntracking_code: String should match pattern '^[A-Z]{2}\\d{9}BR$'", True)


## 5. Resultados de ferramentas também com contrato

O retorno de `get_order` é um `OrderInfo` (Pydantic) — serializado com `model_dump_json()`. Benefícios:
* o modelo recebe sempre o mesmo formato (menos confusão, menos tokens desperdiçados);
* você controla o que **não** vai para o modelo (dados pessoais, campos internos) com `exclude`/modelos de saída separados;
* testes podem validar o retorno de cada ferramenta contra o contrato.

In [9]:
class OrderInfoForLLM(OrderInfo):
    model_config = ConfigDict(json_schema_extra={"description": "Visão do pedido exposta ao modelo"})
    customer: str = Field(exclude=True)          # PII não vai para o contexto do LLM

print(f'OrderInfoForLLM(**ORDERS["456"]).model_dump_json(): {OrderInfoForLLM(**ORDERS["456"]).model_dump_json()}')

OrderInfoForLLM(**ORDERS["456"]).model_dump_json(): {"order_id":"456","total":"59.00","status":"shipped","refunded":"0"}


## 6. Checklist

| ✔ | prática |
|---|---|
| ☐ | Um modelo de argumentos por ferramenta, com `extra="forbid"` e descrições como instrução |
| ☐ | Docstring da função = descrição da ferramenta (quando usar, pré-condições) |
| ☐ | `strict: true` no provedor para garantir forma; Pydantic local para regras de negócio |
| ☐ | Guardrails em validators, alimentados por **contexto de execução** (usuário, limites, estado real) — nunca só pelo prompt |
| ☐ | `execute` nunca levanta: erro vira `tool_result` com `is_error=True` e mensagem acionável |
| ☐ | Limite de passos no loop; logue cada chamada e cada erro de validação |
| ☐ | Resultados com contrato; excluir PII e campos internos do que volta para o modelo |
| ☐ | Ações irreversíveis (pagamento, exclusão) com limite + escalonamento humano |

➡️ Próximo: [`03_enum_routing_and_guardrails.ipynb`](03_enum_routing_and_guardrails.ipynb)